# 06 — 商空间距离与主题网络

**BWV861 音乐形式化分析系统** · Step 6

计算 d_ℚ 商空间距离、S_m 主题相似度，构建主题网络 G_m。

In [4]:
import sys, os, pickle
PROJECT_ROOT = r"e:\大三\Cline Projects\bien_project"
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

# 加载过滤后的主题数据
with open("data/themes_filtered.pkl", "rb") as f:
    filtered = pickle.load(f)
occurrences = filtered["occurrences"]
strict_types = filtered["strict_types"]
symmetric_families = filtered["symmetric_families"]
gains = filtered.get("gains", {})
importance = filtered.get("importance", {})

from music_analysis.distance import (
    build_distance_matrix, compute_scale_parameter,
    theme_similarity, save_similarity,
)
from music_analysis.network import (
    build_theme_network, compute_direct_breakage,
    get_network_stats, save_network,
)

print(f"✅ 已加载过滤后数据: {len(occurrences)} 次出现, "
      f"{len(strict_types)} 个严格类型, {len(symmetric_families)} 个对称族")

✅ 已加载过滤后数据: 268 次出现, 52 个严格类型, 52 个对称族


In [5]:
# 距离矩阵
print()
dist_matrix, type_ids = build_distance_matrix(strict_types)

# 尺度参数
ell = compute_scale_parameter(dist_matrix)
print(f"ℓ = {ell:.4f}")

# 相似度矩阵
sim_matrix = theme_similarity(dist_matrix, ell)

# 展示部分相似度矩阵
print(f"\n相似度矩阵 ({len(type_ids)}×{len(type_ids)}): "
      f"min={sim_matrix.min():.3f}, max={sim_matrix.max():.3f}, "
      f"mean={sim_matrix.mean():.3f}")

save_similarity(dist_matrix, sim_matrix, type_ids, ell)


正在计算 52×52 商空间距离矩阵...
✅ 距离矩阵计算完成
ℓ = 0.3263

相似度矩阵 (52×52): min=0.169, max=1.000, mean=0.397
✅ 相似度数据已保存: data/similarity_matrix.pkl (ℓ=0.3263)


In [6]:
# 主题网络
G = build_theme_network(strict_types, symmetric_families, occurrences)
breakage = compute_direct_breakage(G)
stats = get_network_stats(G)

print(f"\n网络统计:")
for k, v in stats.items():
    print(f"  {k}: {v}")

save_network(G, breakage)

# Top-5 最重要主题类型
sorted_b = sorted(breakage.items(), key=lambda x: -x[1])[:5]
print(f"\nTop-5 B_direct:")
for tid, b in sorted_b:
    n_occ = len(strict_types[tid])
    print(f"  T_{tid}: B_direct={b:.3f}, 出现{n_occ}次")

主题网络: 52 节点, 161 边
  密度: 0.1214
  边权: min=0.062, max=0.500, mean=0.101

网络统计:
  nodes: 52
  edges: 161
  density: 0.12141779788838612
  avg_degree: 6.1923076923076925
  max_degree: 12
  components: 1
✅ 网络数据已保存: data/theme_network.pkl

Top-5 B_direct:
  T_1: B_direct=1.375, 出现11次
  T_2: B_direct=1.375, 出现11次
  T_4: B_direct=1.250, 出现10次
  T_5: B_direct=1.125, 出现9次
  T_3: B_direct=1.062, 出现10次
